# 01 - MSFT: data, features, baselines

Prototype the whole pipeline here, then move finished code into `src/stockpred/`.
Rule: every number that ends up on the resume must come from output in this notebook
or from `outputs/metrics.csv`.

**Question:** can simple models predict MSFT's next-day direction better than trivial baselines,
after transaction costs?

In [1]:
# Imports and settings.
# Needed: numpy, pandas, yfinance, matplotlib, scikit-learn, xgboost.
# Set a global random seed so reruns match.

import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier


## 1. Data

- Stock: MSFT. Reasons: liquid, long history, calmer than NVDA/TSLA, and different from the
  AAPL used in the KPI Anomaly Detection project. Note the survivorship bias (a hindsight pick).
- Download daily bars from 2010-01-01 with adjusted prices.
- Flatten yfinance's MultiIndex columns if present.
- Inspect: `.info()`, date range, missing days, NaNs, zero-volume days.
- Plot the close price. Any split-sized jumps mean the data is unadjusted.

In [15]:
# Download MSFT here
from tarfile import data_filter


stock = yf.Ticker("MSFT")

df = stock.history(start="2010-01-01")

df.head(10)

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2010-01-04 00:00:00-05:00,22.739177,23.095637,22.716898,22.984243,38409100,0.0,0.0
2010-01-05 00:00:00-05:00,22.909975,23.095631,22.754023,22.991663,49749600,0.0,0.0
2010-01-06 00:00:00-05:00,22.932249,23.080774,22.664905,22.850561,58182400,0.0,0.0
2010-01-07 00:00:00-05:00,22.746594,22.798579,22.419840,22.612923,50559700,0.0,0.0
2010-01-08 00:00:00-05:00,22.486680,22.932254,22.456975,22.768877,51197400,0.0,0.0
2010-01-11 00:00:00-05:00,22.806012,22.843144,22.367864,22.479258,68754700,0.0,0.0
2010-01-12 00:00:00-05:00,22.390129,22.575785,22.211899,22.330719,65912100,0.0,0.0
2010-01-13 00:00:00-05:00,22.471831,22.664914,22.286175,22.538668,51863500,0.0,0.0
2010-01-14 00:00:00-05:00,22.508957,23.095631,22.471826,22.991663,63228100,0.0,0.0


In [18]:
# Data quality checks here

print(df.shape)

print(df.isnull().sum())

print(df.duplicated().sum())

print(df.describe())

print(df.info())

print(df.columns)


(4231, 7)
Open            1
High            1
Low             1
Close           1
Volume          0
Dividends       0
Stock Splits    0
dtype: int64
0
              Open         High          Low        Close        Volume  \
count  4230.000000  4230.000000  4230.000000  4230.000000  4.231000e+03   
mean    162.695923   164.252373   161.084617   162.734154  3.520491e+07   
std     154.455099   155.882793   152.965411   154.471413  2.046510e+07   
min      17.305127    17.477504    17.035320    17.245165  5.855900e+06   
25%      32.121285    32.698054    31.807196    32.469630  2.207020e+07   
50%      90.347941    91.260930    89.632814    90.609509  2.940580e+07   
75%     277.279020   279.749139   274.048543   277.202324  4.286580e+07   
max     549.795235   551.150024   543.059998   543.450012  3.193179e+08   

         Dividends  Stock Splits  
count  4231.000000        4231.0  
mean      0.007343           0.0  
std       0.064638           0.0  
min       0.000000           0.0 

## 2. Features and target

Rule: a feature on row t uses data up to the close of day t only.

- Daily returns from the adjusted close.
- Keep the feature set small (8 to 12). With ~4,000 rows, more features means more overfitting.
- Features: lagged returns (1, 2, 5, 10, 21), rolling volatility (5, 21, 63), RSI(14),
  volume z-score, distance from 50- and 200-day moving averages.
- Target: 1 if the next day's return > 0, else 0. Keep `next_return` separately for the backtest.
- Drop warm-up rows (200-day window) and the last row.
- Check the class balance of the target (expect about 53% up).

In [ ]:
# Build features and target here


## 3. Leakage check

Alter prices after some date t and confirm that the features on rows <= t do not change.
Then confirm the target is the only column that looks forward. Write this as a function,
because it becomes the first pytest test.

In [5]:
# Leakage check here

## 4. Baselines

Compute on the same test periods the models will use:
- Always-up (accuracy ~ the up-fraction, AUC = 0.5)
- Previous-day direction
- Buy-and-hold, with the same cost model

These set the bar. Write down the numbers. Buy-and-hold on MSFT since 2010 is strong
(survivorship), so a long/flat strategy that is out of the market part of the time faces a
high bar on raw return. Compare risk-adjusted numbers (Sharpe, drawdown) too.

In [6]:
# Baselines here

## 5. Walk-forward prototype

- Hold out an initial ~3-year training period, then 8 consecutive test blocks of ~6 months.
  Expanding training window, split by date, small gap between train and test.
- Logistic regression (scaler fit on the train window only), then XGBoost (shallow, regularised).
- Per fold: AUC and accuracy. A fold has only ~125 rows, so per-fold numbers are noisy.
  Judge on the pooled out-of-sample predictions.
- Strategy: long if P(up) > 0.5, else flat. Subtract 5 bps on each position change.
- Metrics: net Sharpe (sqrt(252)), max drawdown, equity curve.
- Sanity checks: a shuffled-target model should score AUC ~ 0.5, and an oracle should score 1.0.
- Bootstrap a confidence interval for pooled AUC and Sharpe.

In [7]:
# Fold construction here

In [8]:
# Fit, predict, score here

In [9]:
# Strategy returns, costs, equity curves here

In [10]:
# Bootstrap confidence intervals here

## 6. Findings

Fill in after running. Cover:
- Did either model beat the baselines in pooled AUC? By how much, and does the confidence interval exclude 0.5?
- What happened to the edge after costs, and versus buy-and-hold?
- Which features mattered (feature importances here, SHAP later)?
- What are the limits? (one stock picked with hindsight, one cost assumption, daily bars only, small sample)

If the honest result is "barely better than the baseline", say that. It is a defensible resume story.